# # M16 - Evaluation and production: from notebook to a service you can defend
#
# A model that scores well in a notebook is a hypothesis. Production asks harder
# questions: How precise is that score? Is the probability calibrated? Has the
# input distribution moved? Does the guard hold? Can you reproduce the artifact
# byte for byte? What latency does a caller actually see?
#
# **Prerequisites.** M04-M08 (training and fine-tuning) and basic statistics
# (mean, proportion, confidence interval).
#
# **Study time.** About 75 minutes.
#
# This notebook trains a small churn classifier with NumPy, then treats it like a
# deployed system: confidence intervals, calibration, drift, adversarial input,
# model card, artifact manifest, and a loopback HTTP endpoint. Every number below
# is measured in this run; nothing is copied from a benchmark.

# ## Learning objectives
#
# 1. Report a metric with an interval, not a single decimal.
# 2. Distinguish discrimination (AUC) from calibration (ECE, Brier).
# 3. Measure drift with PSI and explain why "hold" and "investigate" differ.
# 4. Evaluate an input guard on attacks **and** benign traffic.
# 5. Generate a model card from measurements and audit its claims.
# 6. Verify an artifact manifest and detect tampering.
# 7. Measure p50/p95 latency against a real loopback server.

In [1]:
from pathlib import Path
import json
import sys
import tempfile
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "projects").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "projects" / "production"))

from eval_core import artifacts, calibration, cards, drift, guard, metrics
from eval_core import data as eval_data

DATA = eval_data.ensure_data()
print("data directory:", DATA)
print("manifest status:", eval_data.manifest_status())

data directory: E:\AITopics\data\production
manifest status: {'seed': 20261104, 'files': 14, 'bytes': 201062, 'mismatched': []}


# ## 1. A model small enough to inspect
#
# The churn table has numeric and categorical columns. We standardise the numeric
# columns with **training** statistics, one-hot the categorical ones, and fit
# logistic regression with plain gradient descent. No scikit-learn, no black box:
# the point is the evaluation, not the model.
#
# The split is 60/20/20. The validation split selects the calibration temperature;
# the test split is touched once, at the end.

In [2]:
customers = pd.read_csv(DATA / "customers" / "customers.csv")
NUMERIC = ["tenure_months", "monthly_charges", "support_tickets", "usage_gb"]
CATEGORICAL = ["contract_type", "region"]
CATEGORIES = {column: sorted(customers[column].unique()) for column in CATEGORICAL}

rng = np.random.default_rng(0)
order = rng.permutation(len(customers))
train_idx, val_idx, test_idx = np.split(order, [int(0.6 * len(order)), int(0.8 * len(order))])
train, val, test = customers.iloc[train_idx], customers.iloc[val_idx], customers.iloc[test_idx]
print("rows train/val/test:", len(train), len(val), len(test))
print("churn rate:", round(float(train["churned"].mean()), 3))


def design(frame, mean=None, std=None):
    if mean is None:
        mean = frame[NUMERIC].mean().values
        std = frame[NUMERIC].std(ddof=0).values
    blocks = [((frame[NUMERIC].values - mean) / std)]
    for column in CATEGORICAL:
        blocks.append(np.column_stack([
            (frame[column].values == value).astype(float) for value in CATEGORIES[column]
        ]))
    return np.hstack(blocks), mean, std


X_train, MEAN, STD = design(train)
X_val, _, _ = design(val, MEAN, STD)
X_test, _, _ = design(test, MEAN, STD)
y_train = train["churned"].values.astype(float)
y_val = val["churned"].values.astype(float)
y_test = test["churned"].values.astype(float)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


weights = np.zeros(X_train.shape[1])
bias = 0.0
for _ in range(600):
    probs = sigmoid(X_train @ weights + bias)
    gradient = X_train.T @ (probs - y_train) / len(y_train) + 1e-3 * weights
    weights -= 0.5 * gradient
    bias -= 0.5 * float((probs - y_train).mean())

val_probs = sigmoid(X_val @ weights + bias)
test_probs = sigmoid(X_test @ weights + bias)
print("features:", X_train.shape[1], "| weights learned:", np.round(weights, 3)[:5], "...")
print("validation AUC:", round(metrics.roc_auc(y_val, val_probs), 3))
print("test AUC:", round(metrics.roc_auc(y_test, test_probs), 3))

rows train/val/test: 1112 371 371
churn rate: 0.309
features: 11 | weights learned: [-0.861  0.052  0.239 -0.214 -0.499] ...
validation AUC: 0.731
test AUC: 0.778


# ## 2. A number without an interval is a rumour
#
# Accuracy on 371 test rows has sampling error. The Wilson interval is appropriate
# for a proportion; the bootstrap gives an interval for a statistic that is not a
# simple proportion. Both are computed from this run's predictions.
#
# Read the interval width before comparing two models. If the intervals overlap
# heavily, the "winner" may be noise. For paired predictions, use a paired
# bootstrap on the per-row difference instead of comparing two independent
# intervals.

In [3]:
predictions = (test_probs >= 0.5).astype(int)
correct = int((predictions == y_test).sum())
accuracy = metrics.accuracy(y_test, predictions)
low, high = metrics.wilson_interval(correct, len(y_test))
mae = np.abs(test_probs - y_test)
observed, boot_low, boot_high = metrics.bootstrap_ci(mae, n_boot=1000, seed=0)

print(f"accuracy: {accuracy:.3f} ({correct}/{len(y_test)})")
print(f"Wilson 95% interval: [{low:.3f}, {high:.3f}]")
print(f"mean absolute probability error: {observed:.3f} [{boot_low:.3f}, {boot_high:.3f}]")
print("confusion:", metrics.confusion(y_test, predictions))
print("precision/recall/F1:", {key: round(value, 3) for key, value in metrics.precision_recall_f1(y_test, predictions).items()})

accuracy: 0.790 (293/371)
Wilson 95% interval: [0.745, 0.828]
mean absolute probability error: 0.329 [0.308, 0.350]
confusion: {'tp': 48, 'fp': 25, 'fn': 53, 'tn': 245}
precision/recall/F1: {'precision': 0.658, 'recall': 0.475, 'f1': 0.552}


# ## 3. Calibration: does 0.8 mean 80%?
#
# A model can rank well and still be badly calibrated. If every prediction of 0.8
# corresponds to a 55% churn rate, a decision rule that treats 0.8 as "likely" is
# wrong even though AUC looks fine.
#
# Temperature scaling fits one parameter on the validation split and applies it to
# test probabilities. On this run it does **not** improve test ECE. That is a
# result, not a failure: a one-parameter correction cannot fix a misspecified
# model, and fitting it on test would be leakage. The honest conclusion is "keep
# the uncalibrated probabilities and widen the decision threshold analysis", not
# "calibration always helps".

In [4]:
ece_before = calibration.expected_calibration_error(y_test, test_probs)
brier_before = calibration.brier(y_test, test_probs)
fit = calibration.fit_temperature(val_probs, y_val)
calibrated_probs = calibration.apply_temperature(test_probs, fit["temperature"])
ece_after = calibration.expected_calibration_error(y_test, calibrated_probs)
brier_after = calibration.brier(y_test, calibrated_probs)

rows = [
    {"stage": "raw", "ECE": round(ece_before, 3), "Brier": round(brier_before, 3)},
    {"stage": f"temperature {fit['temperature']:.2f}", "ECE": round(ece_after, 3), "Brier": round(brier_after, 3)},
]
print(metrics.table(rows, ["stage", "ECE", "Brier"]))

reliability_rows = calibration.reliability(y_test, test_probs, n_bins=5)
print("\nreliability (5 bins):")
for row in reliability_rows:
    print("  ", {key: (round(value, 3) if isinstance(value, float) else value) for key, value in row.items()})

comparison = metrics.paired_bootstrap_diff(mae, np.abs(calibrated_probs - y_test), n_boot=1000, seed=0)
print("\npaired difference (calibrated - raw) MAE:", {key: round(value, 4) for key, value in comparison.items()})

| stage            | ECE   | Brier |
| ---------------- | ----- | ----- |
| raw              | 0.067 | 0.157 |
| temperature 1.10 | 0.075 | 0.158 |

reliability (5 bins):
   {'bin': '0.0-0.2', 'n': 143, 'confidence': 0.117, 'observed': 0.098, 'gap': 0.02}
   {'bin': '0.2-0.4', 'n': 109, 'confidence': 0.299, 'observed': 0.229, 'gap': 0.069}
   {'bin': '0.4-0.6', 'n': 75, 'confidence': 0.487, 'observed': 0.36, 'gap': 0.127}
   {'bin': '0.6-0.8', 'n': 37, 'confidence': 0.697, 'observed': 0.811, 'gap': 0.113}
   {'bin': '0.8-1.0', 'n': 7, 'confidence': 0.837, 'observed': 0.714, 'gap': 0.123}

paired difference (calibrated - raw) MAE: {'difference': -0.0097, 'low': -0.011, 'high': -0.0084, 'includes_zero': 0}


# Note what the paired bootstrap says: the calibrated model is slightly **worse**
# on this test split, and the interval excludes zero. A team that shipped
# temperature scaling because "calibration is best practice" would have made the
# service worse. The measurement is the argument.

# ## 4. Drift: the world moves
#
# PSI compares a training distribution with a live window. The usual reading:
# below 0.1 stable, 0.1-0.25 moderate, above 0.25 significant. These thresholds are
# conventions, not laws; what matters is that the report names the feature, the
# statistic, and a decision.
#
# Two live weeks are compared against the same training rows. Week 1 should hold;
# week 2 should trigger an investigation.

In [5]:
DRIFT_NUMERIC = ["monthly_charges", "usage_gb", "support_tickets"]
DRIFT_CATEGORICAL = ["region"]
train_records = train.to_dict("records")

drift_rows = []
for name in ("drift_live_w1.csv", "drift_live_w2.csv"):
    live = pd.read_csv(DATA / "customers" / name)
    report = drift.drift_report(train_records, live.to_dict("records"), DRIFT_NUMERIC, DRIFT_CATEGORICAL)
    drift_rows.append({
        "window": name.replace("drift_live_", "").replace(".csv", ""),
        "decision": report["decision"],
        "worst": report["worst"],
        "significant": ", ".join(report["significant"]) or "-",
    })
    print(name, "->", report["decision"])
    for feature in report["features"]:
        print(f"   {feature['feature']:16s} PSI={feature['psi']:.3f} {feature['status']}")

print()
print(metrics.table(drift_rows, ["window", "decision", "worst", "significant"]))

drift_live_w1.csv -> hold
   monthly_charges  PSI=0.065 stable
   usage_gb         PSI=0.168 moderate
   support_tickets  PSI=0.097 stable
   region           PSI=0.094 stable
drift_live_w2.csv -> investigate
   monthly_charges  PSI=0.223 moderate
   usage_gb         PSI=1.171 significant
   support_tickets  PSI=1.212 significant
   region           PSI=0.492 significant

| window | decision    | worst           | significant                       |
| ------ | ----------- | --------------- | --------------------------------- |
| w1     | hold        | usage_gb        | -                                 |
| w2     | investigate | support_tickets | usage_gb, support_tickets, region |


# Week 2 moves `usage_gb`, `support_tickets`, and `region` at once. A single
# feature crossing a threshold can be noise; several moving together usually means
# an upstream change, a new customer segment, or a broken pipeline. The right
# response is to inspect the data and the schema before retraining.

# ## 5. Guardrails are measured on both sides
#
# An input guard that blocks everything has perfect attack recall and is useless.
# Report attack recall and the benign false-positive rate together. The cases file
# contains injections, encoded payloads, path traversal, malformed JSON, and benign
# requests.
#
# This guard is a regex screen. It will miss paraphrases and encodings it does not
# know. The measured misses are listed below and are the most useful part of the
# table.

In [6]:
cases = json.loads((DATA / "adversarial" / "cases.json").read_text(encoding="utf-8"))
guard_report = guard.evaluate_cases(cases)
print("attacks:", guard_report["attacks"], "| blocked:", guard_report["attacks_blocked"],
      "| recall:", round(guard_report["recall"], 3))
print("benign:", guard_report["benign"], "| allowed:", guard_report["benign_allowed"],
      "| false positives:", round(guard_report["false_positive_rate"], 3))
print("families caught:", ", ".join(guard_report["families"]))
print("\nmissed attacks:")
for row in guard_report["rows"]:
    if not row["correct"]:
        print("  ", row["id"], row["kind"], "->", row["reason"])

attacks: 16 | blocked: 12 | recall: 0.75
benign: 12 | allowed: 12 | false positives: 0.0
families caught: limits, malformed_json, path_traversal, prompt_injection, schema_violation, unicode_control, unsafe_output

missed attacks:
   a13 encoded_injection -> accepted
   a14 paraphrase_injection -> accepted
   a15 prompt_extraction -> accepted
   a16 zero_width -> accepted


# ## 6. A model card that can be audited
#
# The card is rendered from a metadata dictionary and a metrics dictionary, then
# audited: every number in the prose must trace back to a measured metric (or be a
# declared count, size, or date). This does not prove the card is complete, but it
# makes an invented number visible.
#
# The artifact digest is the SHA-256 of the serialised model file, so the card and
# the artifact can be checked together.

In [7]:
model_payload = {
    "features": NUMERIC + CATEGORICAL,
    "categories": CATEGORIES,
    "mean": MEAN.tolist(),
    "std": STD.tolist(),
    "weights": weights.tolist(),
    "bias": bias,
}
model_bytes = json.dumps(model_payload, sort_keys=True, indent=2).encode("utf-8")
digest = artifacts.sha256_bytes(model_bytes)

card_meta = {
    "name": "Northwind churn risk",
    "version": "1.0",
    "type": "logistic regression (NumPy, gradient descent)",
    "owner": "AITopics teaching team",
    "created": "2026-11-05",
    "digest": digest,
    "intended_use": "Rank existing customers for a retention review. Output is a probability, not a decision.",
    "out_of_scope": "Do not use for credit, employment, housing, or any eligibility decision. Do not score people who have not opted in.",
    "training_data": "Synthetic Northwind customer table generated offline by the course data script.",
    "limitations": [
        "Trained on synthetic data; the relationships are planted, not observed.",
        "One random split; no temporal validation.",
        "Probabilities are not calibrated for any real population.",
    ],
    "ethical": [
        "Churn scores can encode unfairness through proxy features such as region.",
        "A retention intervention has a cost; a false positive is not free.",
    ],
    "licence": "Teaching material; synthetic data, no personal data.",
}
card_metrics = {
    "test accuracy": round(accuracy, 3),
    "test ROC AUC": round(metrics.roc_auc(y_test, test_probs), 3),
    "test ECE": round(ece_before, 3),
    "test Brier": round(brier_before, 3),
    "guard attack recall": round(guard_report["recall"], 3),
    "guard false positive rate": round(guard_report["false_positive_rate"], 3),
}
card = cards.render_model_card(card_meta, card_metrics)
validation = cards.validate_card(card)
audit = cards.claims_audit(card, card_metrics, extra_allowed=[len(test)])
print("card sections valid:", validation["ok"], "| missing:", validation["missing"])
print("unsupported numbers:", audit["unsupported"])
print("\n" + card[:1200])

card sections valid: True | missing: []
unsupported numbers: []

# Model card: Northwind churn risk

## Model details
- Version: 1.0
- Type: logistic regression (NumPy, gradient descent)
- Owner: AITopics teaching team
- Created: 2026-11-05
- Artifact digest: `ec36c11d3314da78cd3e0e5e629a902b32fab750a63f078ea0c702e05079cbee`

## Intended use
Rank existing customers for a retention review. Output is a probability, not a decision.

## Out-of-scope use
Do not use for credit, employment, housing, or any eligibility decision. Do not score people who have not opted in.

## Training data
Synthetic Northwind customer table generated offline by the course data script.

## Evaluation
Measured on the held-out test split, which was not used for training or model selection.

| Metric | Value |
| --- | --- |
| test accuracy | 0.790 |
| test ROC AUC | 0.778 |
| test ECE | 0.067 |
| test Brier | 0.157 |
| guard attack recall | 0.750 |
| guard false positive rate | 0.000 |

## Limitations
- Trained on 

# ## 7. Artifacts: hash it or it did not happen
#
# A manifest is a list of file names and SHA-256 digests. Verification answers one
# question: is the artifact I am about to serve the artifact I evaluated? A model
# file without a digest is an unversioned dependency.
#
# The tamper check below appends one byte to a copy and confirms the manifest
# catches it.

In [8]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
model_path = out_dir / "churn_model.json"
card_path = out_dir / "model_card.md"
model_path.write_bytes(model_bytes)
card_path.write_text(card, encoding="utf-8")
manifest_path = artifacts.write_manifest(
    out_dir / "manifest.json",
    {"churn_model.json": model_path, "model_card.md": card_path},
    metadata={"created": "2026-11-05", "test_accuracy": round(accuracy, 3)},
)
print("manifest:", manifest_path)
print("verification problems:", artifacts.verify_manifest(manifest_path))

with tempfile.TemporaryDirectory() as tmp:
    tampered = Path(tmp) / "churn_model.json"
    (Path(tmp) / "model_card.md").write_bytes(card_path.read_bytes())
    broken = artifacts.write_manifest(
        Path(tmp) / "manifest.json",
        {"churn_model.json": model_path, "model_card.md": card_path},
    )
    tampered.write_bytes(model_bytes + b" ")
    print("after tampering:", artifacts.verify_manifest(broken))

manifest: outputs\manifest.json
verification problems: []
after tampering: ['digest mismatch: churn_model.json']


# ## 8. Serve it and measure what a caller sees
#
# The endpoint below is a real HTTP server on `127.0.0.1`, not a mock. Each request
# parses JSON, validates the schema, computes the model score, and returns JSON.
# Forty requests give a latency distribution; p50 and p95 are reported separately
# because the mean hides the tail that users complain about.
#
# This is a teaching server: one process, no auth, no rate limit, no TLS, no
# concurrency control on the model. Those are production requirements, not
# optional polish.

In [9]:
def score_record(record):
    frame = pd.DataFrame([record])
    missing = [name for name in NUMERIC + CATEGORICAL if name not in frame.columns]
    if missing:
        raise ValueError(f"missing fields: {missing}")
    matrix, _, _ = design(frame, MEAN, STD)
    return float(sigmoid(matrix @ weights + bias)[0])


class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        try:
            length = int(self.headers.get("Content-Length", "0"))
            payload = json.loads(self.rfile.read(length) or b"{}")
            probability = score_record(payload)
            body = json.dumps({"churn_probability": round(probability, 4)}).encode("utf-8")
            status = 200
        except (ValueError, KeyError, json.JSONDecodeError) as error:
            body = json.dumps({"error": str(error)}).encode("utf-8")
            status = 400
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass


server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
port = server.server_address[1]
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print("listening on http://127.0.0.1:", port)

raw_sample = test.iloc[0][NUMERIC + CATEGORICAL].to_dict()
sample = {key: value.item() if hasattr(value, "item") else value for key, value in raw_sample.items()}
latencies = []
statuses = []
try:
    for _ in range(40):
        started = time.perf_counter()
        request = urllib.request.Request(
            f"http://127.0.0.1:{port}/score",
            data=json.dumps(sample).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(request, timeout=2.0) as response:
            statuses.append(response.status)
            body = json.loads(response.read())
        latencies.append((time.perf_counter() - started) * 1000)
finally:
    server.shutdown()
    server.server_close()
    thread.join(timeout=2)

latencies = sorted(latencies)
p50 = latencies[len(latencies) // 2]
p95 = latencies[int(0.95 * len(latencies)) - 1]
print("statuses:", sorted(set(statuses)), "| responses:", len(latencies))
print(f"latency ms: p50={p50:.2f} p95={p95:.2f} max={latencies[-1]:.2f}")
print("sample response:", body)

listening on http://127.0.0.1: 61291


statuses: [200] | responses: 40
latency ms: p50=15.53 p95=16.72 max=29.60
sample response: {'churn_probability': 0.5152}


# ## 9. What is still missing before this is a service
#
# * **Authentication and authorisation.** Every caller is currently anonymous.
# * **Rate limits and quotas.** One client can saturate the process.
# * **Structured logs with request ids.** The current handler suppresses logs.
# * **Model and schema versioning in the response.** A caller cannot tell which
#   model answered.
# * **Monitoring.** Latency, error rate, drift, and guard blocks need dashboards
#   and alerts with owners.
# * **Rollback.** A digest and a manifest are necessary but not sufficient; you
#   need a previous artifact to roll back to.
# * **Privacy.** Churn features can be personal data; retention and deletion rules
#   apply before the first request.
#
# ## 10. Validation

In [10]:
checks = []
checks.append(("model discriminates", metrics.roc_auc(y_test, test_probs) > 0.7))
checks.append(("accuracy interval covers the point estimate", low <= accuracy <= high))
checks.append(("calibration error is small", 0 < ece_before < 0.2))
checks.append(("week 1 drift holds", drift_rows[0]["decision"] == "hold"))
checks.append(("week 2 drift triggers investigation", drift_rows[1]["decision"] == "investigate"))
checks.append(("guard keeps benign traffic", guard_report["false_positive_rate"] == 0.0))
checks.append(("guard catches most attacks", guard_report["recall"] >= 0.7))
checks.append(("model card is complete", validation["ok"]))
checks.append(("model card claims are traceable", audit["ok"]))
checks.append(("manifest verifies", artifacts.verify_manifest(manifest_path) == []))
checks.append(("server answered every request", len(statuses) == 40 and set(statuses) == {200}))
checks.append(("p95 latency under 250 ms", p95 < 250))
for label, ok in checks:
    print(("ok - " if ok else "FAIL - ") + label)
assert all(ok for _, ok in checks), "M16 validation failed"
print("\nM16 validation passed.")

ok - model discriminates
ok - accuracy interval covers the point estimate
ok - calibration error is small
ok - week 1 drift holds
ok - week 2 drift triggers investigation
ok - guard keeps benign traffic
ok - guard catches most attacks
ok - model card is complete
ok - model card claims are traceable
ok - manifest verifies
ok - server answered every request
ok - p95 latency under 250 ms

M16 validation passed.


# ## Practice
#
# 1. Replace the logistic model with a shallow NumPy MLP. Report the same
#    intervals. Did the extra capacity move the interval or just the point?
# 2. Find a threshold that keeps precision above 0.7. Report recall and the
#    confusion matrix at that threshold.
# 3. Add a schema version to the model payload and reject requests with a
#    different version. Measure the new error path.
# 4. Extend the guard with a normalisation step for base64 and zero-width
#    characters. Re-run `evaluate_cases` and report both recall and false
#    positives.
# 5. Simulate a week-3 window with a new region value. What does PSI report, and
#    what should the service do before retraining?
#
# ## Common mistakes
#
# * **Reporting one accuracy number with no interval.**
# * **Fitting calibration on the test split.** It looks better and means nothing.
# * **Treating PSI thresholds as universal.** They are conventions.
# * **Reporting guard recall without false positives.**
# * **Writing a model card by hand.** Generate it, then audit it.
# * **Serving an artifact without a digest.**
# * **Quoting the mean latency.** The tail is what users experience.
#
# ## Knowledge checks
#
# 1. Why is a paired bootstrap better than comparing two independent intervals?
#    *It uses the correlation between predictions on the same rows, so it has more
#    power and answers the right question: which model is better on this sample?*
# 2. What does ECE measure that AUC does not?
#    *Whether predicted probabilities match observed frequencies.*
# 3. Why can a guard with 0.75 recall still be useful?
#    *It raises the cost of an attack and blocks a known family, while the real
#    defence is capability limitation. But the miss rate must be documented.*
# 4. What does a manifest prove?
#    *That the bytes being served match the bytes that were evaluated, assuming
#    the manifest itself is trusted.*
# 5. Why measure p95 instead of the mean?
#    *A mean hides a slow tail; p95 bounds the experience of most callers.*
#
# ## Lab connection
#
# The section lab asks for an evaluation report plus a served endpoint with a
# model card, a manifest, and a drift decision. Reuse `eval_core` and the patterns
# here; do not build a second metrics library.
#
# ## Further reading
#
# * Sculley et al., *Hidden Technical Debt in Machine Learning Systems* (2015).
# * Guo et al., *On Calibration of Modern Neural Networks* (2017).
# * NIST, *AI Risk Management Framework* (AI RMF 1.0).
# * Google, *Model Cards for Model Reporting* (2019).
#
# ---
#
# **Previous:** [M12 - Speech and audio](../speech/m12_speech_and_audio.md)
# **Next:** [M17 - Capstone](../capstone/m17_capstone.md)